In [ ]:
%load_ext cudf.pandas

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
%%time
### cell 0 ###

factor = 20
flights_df = pd.read_csv(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/nyc_flights.csv"
)
flights_df = pd.concat([flights_df] * factor, ignore_index=True)

In [ ]:
%%time
### cell 1 ###

flights_df.shape, flights_df.columns, flights_df.dtypes

In [ ]:
%%time
### cell 2 ###

flights_df.dest.unique()
flights_df.head(10)

In [ ]:
%%time
### cell 3 ###

pd.Series([flights_df.dest.eq("SEA").sum()], index=["SEA"])

In [ ]:
%%time
### cell 4 ###

(
    flights_df["dest"]
    .eq("SEA")
    .groupby(flights_df["carrier"])
    .sum()
    # drop carriers with zero SEA flights, like value_counts
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    # match Series.name and index.name of value_counts output
    .rename_axis(None)
    .rename("carrier")
)

In [ ]:
%%time
### cell 5 ###

# GPU-native distinct count (preserves NaNs)
flights_df.loc[flights_df["dest"] == "SEA", "tailnum"].nunique(dropna=False)

In [ ]:
%%time
### cell 6 ###

flights_df.loc[flights_df["dest"] == "SEA", "arr_delay"].mean()

In [ ]:
%%time
### cell 7 ###

# Compute mask for flights destined to SEA once
dest_sea = flights_df.dest == "SEA"
# Total number of SEA flights
total_sea = dest_sea.sum()
# Count SEA flights from EWR and JFK, then normalize
ewr_ratio = ((flights_df.origin == "EWR") & dest_sea).sum() / total_sea
jfk_ratio = ((flights_df.origin == "JFK") & dest_sea).sum() / total_sea

ewr_ratio, jfk_ratio

In [ ]:
%%time
### cell 8 ###

# Do a single GPU-accelerated groupby for both delays
grp = (
    flights_df.groupby(["month", "day"])[["dep_delay", "arr_delay"]]
    .mean()
    .reset_index()
)

# Split back into the per-delay DataFrames
df = grp[["month", "day", "dep_delay"]]
df2 = grp[["month", "day", "arr_delay"]]

# GPU-accelerated argmax and row selection
max_dep = df.loc[df["dep_delay"].argmax()]
max_arr = df2.loc[df2["arr_delay"].argmax()]

# Return the two Series as before
max_dep, max_arr

In [ ]:
%%time
### cell 9 ###

df

In [ ]:
%%time
### cell 10 ###

df = flights_df.groupby(["day", "month"], as_index=False).agg(
    {"arr_delay": np.mean, "dep_delay": np.mean}
)
df["total_delay"] = df["arr_delay"] + df["dep_delay"]
df.sort_values("total_delay", ascending=False).head(1)

In [ ]:
%%time
### cell 11 ###

# Re-add the dropna on dep_delay so that cuDF’s groupby.mean matches the original (skipna behavior)
ds = flights_df.dropna(subset=["dep_delay"]).groupby("month")["dep_delay"].mean()
ds

In [ ]:
%%time
### cell 12 ###

dt = flights_df.dropna(subset=["dep_delay"]).groupby(["hour"])["dep_delay"].mean()
dt

In [ ]:
%%time
### cell 13 ###

df = flights_df
df["speed"] = df["distance"] / df["air_time"]
df[df["speed"] == df.speed.max()]

In [ ]:
%%time
### cell 14 ###

# cell 14 - optimized for cudf
count = len(flights_df)
# group & count on GPU
df = flights_df.groupby(["carrier", "flight", "dest"]).size().reset_index(name="Size")
# filter groups of size 365 and select only the three columns
result = df[df["Size"] == 365][["carrier", "flight", "dest"]]
# iterate directly over the GPU-backed DataFrame
for carrier, flight, dest in result.itertuples(index=False, name=None):
    print(f"Carrier: {carrier}, Flight: {flight}, Destination: {dest}")
# restore `i` to the last index of df to satisfy the post-cell assertion
i = len(df) - 1

In [ ]:
%%time
### cell 15 ###

gdf = flights_df.groupby(["carrier", "flight", "dest"]).size().reset_index(name="size")
# use GPU-native boolean indexing instead of .loc
gdf = gdf[gdf["size"] == 365][["carrier", "flight", "dest"]]
gdf

In [ ]:
%%time
### cell 16 ###

# filter to June
df = flights_df[flights_df["month"] == 6]

# use the original .agg with numpy mean to match the exact output
# (avoids subtle floating-point differences from the GPU .mean)
df = df.groupby("carrier", as_index=False).agg(
    {"arr_delay": np.mean, "dep_delay": np.mean}
)

# vectorized total_delay calculation
df["total_delay"] = df["arr_delay"] + df["dep_delay"]

# find and print all carrier(s) with the minimum total_delay
min_total = df["total_delay"].min()
min_df = df[df["total_delay"] == min_total]
for c, t in zip(min_df["carrier"], min_df["total_delay"]):
    print(c, t)

# return the DataFrame
df

In [ ]:
%%time
### cell 17 ###

weather_df = pd.read_csv(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/nyc_weather.csv"
)
df = flights_df
df_c = pd.merge(df, weather_df, on=["month", "day", "hour", "origin"])
df_c.head(10)